Recommender system

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [19]:
# Extract CSV's to PD's dataframes
df_financial = pd.read_csv("data/raw/financial_impact.csv")
df_incidents = pd.read_csv("data/raw/incidents_master.csv")
df_market = pd.read_csv("data/raw/market_impact.csv")

df_merged = df_incidents.merge(df_financial, on='incident_id', how='inner').merge(df_market, on='incident_id', how='inner') # master

provided_features = [
    'company_revenue_usd',
    'employee_count',
    'pre_incident_volatility_30d',  # baseline, known before
    'market_cap_at_disclosure',     # judgment call — you decide
    # encoded categoricals:
    #'attack_vector_primary_enc',
    #'industry_primary_enc',
    #'data_type_enc',
    #'attribution_confidence_enc',
]

predict_features = [
    'days_to_price_recovery',
    'total_loss_usd',
    'downtime_hours',
    'abnormal_return_30d',
    'car_0_to_30',
]

df_merged

,incident_id,company_name,company_revenue_usd,country_hq,industry_primary,industry_secondary,employee_count,is_public_company,stock_ticker_x,incident_date,...,p_value_30d,earnings_announcement_within_7d,market_cap_at_disclosure,volume_ratio_disclosure,pre_incident_volatility_30d,post_incident_volatility_30d,days_to_price_recovery,notes,created_at,updated_at
0,2023-0115-001,BitWire Innovations Corp.,2.480619e+10,US,51,NaN,71369,True,BITW,2023-01-15,...,1.00000,True,1.181988e+11,2.4652,0.027705,0.052161,255.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
1,2021-0315-001,Sterling Forge Markets Holdings Inc.,1.398259e+08,US,44-45,NaN,912,True,SFM,2021-03-15,...,0.98240,False,6.489114e+08,3.0973,0.017116,0.027638,324.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
2,2021-1204-001,Sierra Quantum Innovations Group Inc.,6.916977e+08,US,51,NaN,1662,True,SQI,2021-12-04,...,1.00000,False,4.735164e+09,1.5348,0.038209,0.045756,19.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
3,2021-0213-001,Stuttgart Distribution SE,7.334128e+08,DE,44-45,NaN,8169,True,STUT.DE,2021-02-13,...,1.00000,False,2.984412e+09,2.5748,0.027980,0.030972,24.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
4,2025-0529-001,BazaarBrand Stores Co.,3.991403e+10,US,44-45,NaN,776266,True,BAZA,2025-05-29,...,0.38875,False,2.194296e+11,2.6298,0.025212,0.038232,313.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,2022-1206-001,Atlanta Stores Group Inc.,3.739506e+10,US,44-45,NaN,286773,True,ATL3,2022-12-06,...,1.00000,False,1.772076e+11,3.2483,0.020154,0.028128,49.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
325,2024-0823-001,Citadel Care Labs Group Inc.,3.354977e+10,DK,62,NaN,137859,True,CCL.CO,2024-08-23,...,1.00000,False,1.059913e+11,3.5838,0.010241,0.012128,NaN,CEO resignation announced 7 days post-disclosu...,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
326,2022-0508-002,Blackwell Intelligence Group PLC,1.098141e+09,GB,51,21,1885,True,BLAC.L,2022-05-08,...,1.00000,False,6.120250e+09,3.0012,0.010470,0.020021,NaN,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z
327,2025-1112-001,Vista Dynamo Outlets SA,1.207104e+09,FR,44-45,NaN,16078,True,VDO.PA,2025-11-12,...,1.00000,False,3.327991e+09,2.5376,0.030708,0.044332,19.0,NaN,2026-02-12T10:00:00Z,2026-02-12T10:00:00Z


In [20]:
from utils import define_feature_types
num_cols, cat_cols = define_feature_types(df_merged)

In [21]:
df_merged = df_merged[num_cols]
    
# Fit scaler on train only to avoid data leaking
"""scaler = StandardScaler()
df_numerical_scaled = scaler.fit_transform(df_numerical)
df_numerical_scaled = pd.DataFrame(df_numerical_scaled, columns=df_numerical.columns)"""

# It is needed?

'scaler = StandardScaler()\ndf_numerical_scaled = scaler.fit_transform(df_numerical)\ndf_numerical_scaled = pd.DataFrame(df_numerical_scaled, columns=df_numerical.columns)'

In [22]:
# Now that all the data is normalized, we can fix our target
target = df_merged[3:4]

if 3 in df_merged:
    df_merged.drop(index=3, inplace=True) # drop the target row from the data we will correlate with
target = target.squeeze() # to make it a Series instead of a DataFrame, so we can correlate with other rows (users) in the next step

target["employee_count"] = np.nan
target["quality_score"] = np.nan
target


company_revenue_usd             7.334128e+08
employee_count                           NaN
data_compromised_records        3.212150e+05
downtime_hours                           NaN
confidence_tier                 3.000000e+00
quality_score                            NaN
direct_loss_usd                 4.864706e+06
ransom_demanded_usd                      NaN
ransom_paid_usd                          NaN
recovery_cost_usd               1.593636e+06
legal_fees_usd                  1.332120e+06
regulatory_fine_usd                      NaN
insurance_payout_usd            2.496438e+06
total_loss_usd                  7.790462e+06
total_loss_lower_bound          5.394998e+06
total_loss_upper_bound          1.261305e+07
inflation_adjusted_usd          9.243213e+06
price_7d_before                 5.340000e+00
price_disclosure_day            5.430000e+00
price_1d_after                  5.270000e+00
price_7d_after                  5.360000e+00
price_30d_after                 5.370000e+00
volume_avg

In [23]:
similarity = df_merged.corrwith(target, axis = 1, method = 'pearson') # This function calculates everything based on common items (e.g. mean of user is calculated based only on the common items with that other user we correlate with)
similarity

# only keep neighbours with enough shared features
min_shared = 5
valid_neighbours = similarity[
    df_merged.loc[similarity.index].notna().sum(axis=1) >= min_shared
]

In [24]:
topN = 20

similarity.sort_values(ascending = False, inplace = True)
idx = similarity.index[:topN] # get the indices of the top N most similar users
idx

Index([  3,  70,  88, 210,  64, 205, 169, 148,  20, 132, 275, 101,  44, 143,
       250, 185, 144, 178,  43, 267],
      dtype='int64')

In [25]:
def recommend_items(target, similarity, idx):
    # Get the ratings of the top N most similar users
    similar_users_ratings = df_merged.loc[idx]
    outcome_cols = ['days_to_price_recovery', 'total_loss_usd', 
                'downtime_hours', 'abnormal_return_30d']

    """if target == 0:
        print(f"El target {target_id} ya tiene datos en todas las columnas. ¡Nada que recomendar!")
        return pd.Series(dtype=float)"""
    
    neighbor_matrix = df_merged.loc[idx, outcome_cols]
    return neighbor_matrix

def recommend_items_B(df, predict_features, needed_features, k_similar=5):

    from sklearn.metrics import mean_absolute_error

    predictions = []
    actuals = []
    for ftr in predict_features:
        predictions = []
        actuals = []
        if ftr not in df.columns:
            print(f"Feature '{ftr}' not found in DataFrame. Skipping.")
            return
        for idx in df.index:
            actual = df.loc[idx, ftr]
            if pd.isna(actual):
                continue
                
            # remove target from pool
            #df_pool = df_scaled.drop(index=idx)
            #target = df_scaled.loc[idx, similarity_features]
            target = df.loc[idx, needed_features]
            df_pool = df.drop(index=idx)
            
            
            # compute similarity
            similarity = df_pool[needed_features].corrwith(target, axis=1)
            top_idx = similarity.nlargest(k_similar).index
            
            # predict outcome as weighted mean of neighbours
            try:
                weights = similarity[top_idx]
                weights = weights[weights > 0] # only keep positive correlations
                neighbour_outcomes = df.loc[top_idx, ftr].dropna()
                predicted = np.average(neighbour_outcomes, 
                                    weights=weights[neighbour_outcomes.index])
                
                predictions.append(predicted)
                actuals.append(actual)
            except ZeroDivisionError as e:
                continue

        if len(predictions) > 0:
            mae = mean_absolute_error(actuals, predictions)
            print(f"MAE: {mae:.4f} {ftr}")
        else:
            print(f"No predictions made for feature '{ftr}'.")

#recommend_items(target, similarity, idx)

recommend_items_B(df_merged, predict_features, provided_features, k_similar=20)



MAE: 90.0594 days_to_price_recovery
MAE: 102208598.7471 total_loss_usd
MAE: 92.1523 downtime_hours
MAE: 0.0202 abnormal_return_30d
MAE: 0.0202 car_0_to_30
